In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import plot_style
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis_config import BASE_PATH, NUM_AUS_TO_CHOOSE, FOCUS_AUS, CONFIDENCE_THRESHOLD
from dyad_analysis_utils import (
    load_openface_data,
    load_prepared_meetings,
    organize_sessions_by_period,
    select_best_aus_from_correlations,
)

base_path = BASE_PATH
if not base_path.exists():
    raise FileNotFoundError(f"Expected directory not found: {base_path.resolve()}")

meeting_groups, input_paths = load_openface_data(base_path)
print(f"Found {len(input_paths)} session folders, {len(meeting_groups)} meetings")
print(f"NUM_AUS_TO_CHOOSE = {NUM_AUS_TO_CHOOSE}, CONFIDENCE_THRESHOLD = {CONFIDENCE_THRESHOLD}")
print(f"Config FOCUS_AUS = {FOCUS_AUS}")


In [ ]:
df_infant_arr, df_adult_arr, AUs = load_prepared_meetings(meeting_groups)
_, all_periods_data = organize_sessions_by_period(
    df_infant_arr, df_adult_arr, AUs, include_frame=False
)
print(f"Sessions used for ranking: {len(all_periods_data)}")


In [ ]:
selected_aus, all_aus_ranked = select_best_aus_from_correlations(
    all_periods_data, AUs, NUM_AUS_TO_CHOOSE
)

print("All AUs ranked by average rank (lower = more important):")
for au, avg_rank in all_aus_ranked:
    marker = " <--" if au in selected_aus else ""
    print(f"  {au}: {avg_rank:.2f}{marker}")

print(f"\nComputed selection ({len(selected_aus)} AUs): {selected_aus}")
print(f"Config FOCUS_AUS ({len(FOCUS_AUS)} AUs):          {FOCUS_AUS}")
if selected_aus != FOCUS_AUS:
    print("\nWARNING: computed selection differs from analysis_config.FOCUS_AUS — update the config if intentional.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.0))
ranked_aus = [au for au, _ in all_aus_ranked]
avg_ranks = [rank for _, rank in all_aus_ranked]

ax.plot(range(1, len(ranked_aus) + 1), avg_ranks, marker="o")
ax.axvline(x=NUM_AUS_TO_CHOOSE, color="black", linestyle="--", label=f"Chosen: {NUM_AUS_TO_CHOOSE} AUs")
ax.set_xticks(range(1, len(ranked_aus) + 1))
ax.set_xticklabels(plot_style.clean_au_labels(ranked_aus), rotation=90, ha="center", va="top")
ax.set_xlabel("Action Units (ranked)")
ax.set_ylabel("Average rank")
plot_style.ax_title(ax, "Elbow method for AU selection", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plot_style.save_pdf(fig, "au_selection_elbow", PDF_DIR)
plot_style.finish_figure(fig)